In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_27.pickle

In [ ]:
Out.clear()

In [ ]:
%%RecordEvent
%%time
### cell 28 ###

def grab_subset_of_data_40(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how='all', subset=mapper, inplace=True)
    return new_df

def add_year_column_to_dataframes_40(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None):
    if df_2017 is not None:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        df_2017['year'] = '2017'
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022['year'] = '2022'
        df_2021['year'] = '2021'
        df_2020['year'] = '2020'
        df_2019['year'] = '2019'
        df_2018['year'] = '2018'
        return (df_2018, df_2019, df_2020, df_2021, df_2022)

def convert_df_of_counts_to_percentages_40(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = df_combined_percentages[0:1] * 100 / df['year'].eq('2018').sum()
    df_combined_percentages[1:2] = df_combined_percentages[1:2] * 100 / df['year'].eq('2019').sum()
    df_combined_percentages[2:3] = df_combined_percentages[2:3] * 100 / df['year'].eq('2020').sum()
    df_combined_percentages[3:4] = df_combined_percentages[3:4] * 100 / df['year'].eq('2021').sum()
    df_combined_percentages[4:5] = df_combined_percentages[4:5] * 100 / df['year'].eq('2022').sum()
    df_combined_percentages['year'] = ['2018', '2019', '2020', '2021', '2022']
    return df_combined_percentages

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(question_of_interest, include_2017=None):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_40(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_40(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_40(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_40(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_40(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_40(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_40(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_40(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_40(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_40(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_40(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_40(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_40(df_2022, df_2021, df_2020, df_2019, df_2018, df_2017)
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby('year').count().reset_index()
        return (df_combined, df_combined_counts)
question_of_interest_original = 'Which of the following hosted notebook products do you use on a regular basis?'
question_of_interest_new = 'Do you use any of the following hosted notebook products?'
responses_df_2021.columns = responses_df_2021.columns.str.replace(question_of_interest_original, question_of_interest_new, regex=False)
question_of_interest_original_cell40 = 'Which of the following hosted notebook products do you use on a regular basis?'
question_of_interest_new_cell40 = 'Do you use any of the following hosted notebook products?'
responses_df_2020.columns = responses_df_2020.columns.str.replace(question_of_interest_original_cell40, question_of_interest_new_cell40, regex=False)
colab_text_to_replace = 'Google Colab '
colab_new_text = 'Colab Notebooks'
colab_answer = 'Which of the following hosted notebook products do you use on a regular basis?  (Select all that apply) - Selected Choice -  Google Colab '
kaggle_text_to_replace = 'Kaggle Notebooks (Kernels) '
kaggle_new_text = 'Kaggle Notebooks'
kaggle_answer = 'Which of the following hosted notebook products do you use on a regular basis?  (Select all that apply) - Selected Choice -  Kaggle Notebooks (Kernels) '
responses_df_2019_cell10[colab_answer] = responses_df_2019_cell10[colab_answer].str.replace(colab_text_to_replace, colab_new_text, regex=False)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(colab_text_to_replace, colab_new_text, regex=False)
responses_df_2019_cell10[kaggle_answer] = responses_df_2019_cell10[kaggle_answer].str.replace(kaggle_text_to_replace, kaggle_new_text, regex=False)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(kaggle_text_to_replace, kaggle_new_text, regex=False)
question_of_interest_original_cell40 = 'Which of the following hosted notebook products do you use on a regular basis?'
question_of_interest_new_cell40 = 'Do you use any of the following hosted notebook products?'
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(question_of_interest_original_cell40, question_of_interest_new_cell40, regex=False)
colab_text_to_replace_cell40 = 'Google Colab'
colab_new_text_cell40 = 'Colab Notebooks'
colab_answer_cell40 = 'Which of the following hosted notebooks have you used at work or school in the last 5 years? (Select all that apply) - Selected Choice - Google Colab'
kaggle_text_to_replace_cell40 = 'Kaggle Kernels'
kaggle_new_text_cell40 = 'Kaggle Notebooks'
kaggle_answer_cell40 = 'Which of the following hosted notebooks have you used at work or school in the last 5 years? (Select all that apply) - Selected Choice - Kaggle Kernels'
responses_df_2018_cell10[colab_answer_cell40] = responses_df_2018_cell10[colab_answer_cell40].str.replace(colab_text_to_replace_cell40, colab_new_text_cell40, regex=False)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(colab_text_to_replace_cell40, colab_new_text_cell40, regex=False)
responses_df_2018_cell10[kaggle_answer_cell40] = responses_df_2018_cell10[kaggle_answer_cell40].str.replace(kaggle_text_to_replace_cell40, kaggle_new_text_cell40, regex=False)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(kaggle_text_to_replace_cell40, kaggle_new_text_cell40, regex=False)
question_of_interest_original_cell40 = 'Which of the following hosted notebooks have you used at work or school in the last 5 years?'
question_of_interest_new_cell40 = 'Do you use any of the following hosted notebook products?'
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(question_of_interest_original_cell40, question_of_interest_new_cell40, regex=False)
question_of_interest_cell40 = 'Do you use any of the following hosted notebook products?'
(notebooks_df_combined, notebooks_df_combined_counts) = combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(question_of_interest_cell40)
notebooks_df_combined_percentages = convert_df_of_counts_to_percentages_40(notebooks_df_combined, notebooks_df_combined_counts)
notebooks_df_combined_percentages_cell40 = notebooks_df_combined_percentages.loc[:, ['year', 'None', 'Kaggle Notebooks', 'Colab Notebooks']]
df_cell40 = notebooks_df_combined_percentages_cell40.melt(id_vars=['year'], value_vars=['None', 'Kaggle Notebooks', 'Colab Notebooks'])
df_cell40 = df.rename(columns={'variable': ''})
df_cell40.info()

In [ ]:
%%RecordEvent
orig_output = Out.get(5)

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_28.pickle